# Mojito Processing Pipeline

Demonstrates the full TDI data processing pipeline using the `process_pipeline` utility from `MojitoProcessor`.

The pipeline applies the following steps in order:

| Step | Operation | Default |
|------|-----------|------|
| 1 | Band-pass filter | configurable (f_low, f_high), order |
| 2 | Trim edge artefacts | 2.2% from each end |
| 3 | Truncate to working length | configurable |
| 4 | Downsample | configurable target rate |
| 5 | Tukey window | α = 0.025 |

The final cell verifies consistency by comparing the periodogram against the Mojito L1 noise estimate.

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt

from mojito import MojitoL1File
from MojitoProcessor import process_pipeline

# Show pipeline progress at INFO level
logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(message)s",
)

## 1. Load Data

In [ ]:
mojito_data_file = (
    "../Mojito_Data/NOISE_731d_0.25s_L1_source0_0_20251206T220508924302Z.h5"
)

# ── How many days to load (lazy slicing — only reads what is needed) ───────────
load_days = None  # set to None to load the full dataset

with MojitoL1File(mojito_data_file) as f:
    tdi_sampling = f.tdis.time_sampling
    ltt_sampling = f.ltts.time_sampling
    orbit_sampling = f.orbits.time_sampling

    # Consistent sample counts across all data streams
    n_tdi = int(load_days * 86400 * tdi_sampling.fs) if load_days else tdi_sampling.size
    n_ltt = int(load_days * 86400 * ltt_sampling.fs) if load_days else ltt_sampling.size
    n_orbit = (
        int(load_days * 86400 * orbit_sampling.fs) if load_days else orbit_sampling.size
    )

    data = {
        # ── TDI observables ──────────────────────────────────────────────────
        "tdis": {
            "X": f.tdis.x2[:n_tdi],
            "Y": f.tdis.y2[:n_tdi],
            "Z": f.tdis.z2[:n_tdi],
            "A": f.tdis.a2[:n_tdi],
            "E": f.tdis.e2[:n_tdi],
            "T": f.tdis.t2[:n_tdi],
        },
        "fs": tdi_sampling.fs,
        "dt": tdi_sampling.dt,
        "t_tdi": tdi_sampling.t()[:n_tdi],
        # ── Light travel times ───────────────────────────────────────────────
        "ltts": {
            "12": f.ltts.ltt_12[:n_ltt],
            "13": f.ltts.ltt_13[:n_ltt],
            "21": f.ltts.ltt_21[:n_ltt],
            "23": f.ltts.ltt_23[:n_ltt],
            "31": f.ltts.ltt_31[:n_ltt],
            "32": f.ltts.ltt_32[:n_ltt],
        },
        "ltt_derivatives": {
            "12": f.ltts.ltt_derivative_12[:n_ltt],
            "13": f.ltts.ltt_derivative_13[:n_ltt],
            "21": f.ltts.ltt_derivative_21[:n_ltt],
            "23": f.ltts.ltt_derivative_23[:n_ltt],
            "31": f.ltts.ltt_derivative_31[:n_ltt],
            "32": f.ltts.ltt_derivative_32[:n_ltt],
        },
        "ltt_times": ltt_sampling.t()[:n_ltt],
        # ── Spacecraft orbits ────────────────────────────────────────────────
        "orbits": f.orbits.positions[:n_orbit],  # (n_orbit, 3, 3)
        "velocities": f.orbits.velocities[:n_orbit],  # (n_orbit, 3, 3)
        "orbit_times": orbit_sampling.t()[:n_orbit],
        # ── Noise estimates (frequency-domain, not truncated) ────────────────
        "noise_estimates": {
            "xyz": f.noise_estimates.xyz[:],
            "aet": f.noise_estimates.aet[:],
            "freqs": f.noise_estimates.freq_sampling.f(),
        },
        # ── Metadata ─────────────────────────────────────────────────────────
        "metadata": {
            "laser_frequency": f.laser_frequency,
            "pipeline_name": f.pipeline_name,
        },
    }

n_samples = len(data["tdis"]["X"])
duration = n_samples * data["dt"]
print(f"Loaded: {n_samples:,} samples @ {data['fs']} Hz ({duration / 86400:.2f} days)")
print(f"TDI channels: {list(data['tdis'].keys())}")

In [ ]:
from lisaglitch import GapMaskGenerator
from lisagap import GapWindowGenerator
import numpy as np

# Create time array
# dt = 5.0  # seconds
# duration = 86400  # 1 day in seconds
sim_t = data["t_tdi"]

# Define gap configuration
gap_definitions = {
    "planned": {
        "antenna_repointing": {
            "rate_per_year": 26,  # 12 times per year
            "duration_hr": 7,
        }
    },
    "unplanned": {
        #     "hardware_failure": {
        #         "rate_per_year": 4,   # 4 times per year
        #         "duration_hr": 0.5    # 30 minutes each
    },
}


# Create gap mask generator
gap_gen = GapMaskGenerator(
    sim_t=sim_t,
    gap_definitions=gap_definitions,
    treat_as_nan=False,
)

# Wrap with windowing capabilities
window_gen = GapWindowGenerator(gap_gen)

# Define custom tapering per gap type
taper_definitions = {
    "planned": {
        "antenna_repointing": {"lobe_lengths_hr": 3.0},
    },
    "unplanned": {
        #     "platform safe mode": {"lobe_lengths_hr": 1.0},
        #     "HR GRS loss micrometeoroid": {"lobe_lengths_hr": 7.0},
    },
}


# Create window generator for advanced features
window_func = GapWindowGenerator(gap_gen)

# Generate window function
smoothed_mask = window_func.generate_window(
    include_planned=True,
    include_unplanned=False,
    apply_tapering=True,
    taper_definitions=taper_definitions,
)[0]

from scipy.signal import windows

full_mask = windows.tukey(
    len(smoothed_mask), alpha=0.01
)  # Example: Tukey window for the entire dataset

smoothed_mask = full_mask * smoothed_mask
# Apply gaps to a deep-copied dataset (dict has no .deepcopy() method)
from copy import deepcopy

data_w_gaps = deepcopy(data)
for ch in ["X", "Y", "Z", "A", "E", "T"]:
    data_w_gaps["tdis"][ch] = data["tdis"][ch] * smoothed_mask

In [ ]:
plt.plot(data_w_gaps["t_tdi"][::50] / 3600, data_w_gaps["tdis"]["X"][::50], label="X")
plt.plot(
    data_w_gaps["t_tdi"][::50] / 3600,
    np.max(data_w_gaps["tdis"]["X"][::50]) * smoothed_mask[::50],
    label="Smoothed Mask",
)
# plt.xlim([27500, 29000])
plt.xlabel("Time (hours)")
plt.ylabel("Signal")
plt.title("TDI Data with Gaps")
plt.legend()
plt.show()

## 2. Run the Processing Pipeline

All pipeline parameters are configurable here. The pipeline runs in this order:

```
bandpass/highpass filter → downsample → trim → truncate → window
```

We remark that the time-domain data will be normalised by the central frequency of the laser through the processing pipeline. The units are thus dimensionless. 

In [ ]:
# ── Pipeline parameters ───────────────────────────────────────────────────────

# Downsampling parameters
downsample_kwargs = {
    "target_fs": 0.2,  # Hz — target sampling rate (None = no downsampling).
    "kaiser_window": 31.0,  # Kaiser window beta parameter (higher = more aggressive anti-aliasing)
}

# Filter parameters
filter_kwargs = {
    "highpass_cutoff": 5e-5,  # Hz — high-pass cutoff (always applied)
    "lowpass_cutoff": 0.8
    * downsample_kwargs[
        "target_fs"
    ],  # Hz — low-pass cutoff (set None for high-pass only)
    "order": 2,  # Butterworth filter order
}

# Trim parameters
trim_kwargs = {
    "fraction": 0.02,  # Fraction of post-downsample duration trimmed from each end.
    # Total amount of data remaining is (1 - fraction) * N, for N
    # the number of samples after downsampling.
}

# Segmentation parameters
truncate_kwargs = {
    "days": 15.0,  # Segment length in days (splits dataset into 7-day chunks)
}

# Window parameters
window_kwargs = {
    "window": "tukey",  # Window type: 'tukey', 'hann', 'hamming', 'blackman'
    "alpha": 0.0,  # Taper fraction for Tukey window
}
# ─────────────────────────────────────────────────────────────────────────────

processed_segments = process_pipeline(
    data_w_gaps,
    downsample_kwargs=downsample_kwargs,
    filter_kwargs=filter_kwargs,
    trim_kwargs=trim_kwargs,
    truncate_kwargs=truncate_kwargs,
    window_kwargs=window_kwargs,
)

# For the rest of the notebook, use the first segment
sp_0 = processed_segments["segment0"]

In [ ]:
from fractions import Fraction
from scipy import signal as scipy_signal
from scipy.ndimage import binary_closing

# ── Mirror the pipeline parameters ───────────────────────────────────────────
_fs = data["fs"]
_highpass = filter_kwargs["highpass_cutoff"]
_lowpass = filter_kwargs.get("lowpass_cutoff", None)
_order = filter_kwargs.get("order", 2)
_target_fs = downsample_kwargs["target_fs"]
_trim_frac = trim_kwargs["fraction"]
_ratio = Fraction(_target_fs / _fs).limit_denominator(10000)


def _downsample_trim(arr, ratio, trim_n):
    ds = scipy_signal.resample_poly(arr, ratio.numerator, ratio.denominator)
    return ds[trim_n:-trim_n] if trim_n > 0 else ds


_trim_n = int(
    round(
        len(
            scipy_signal.resample_poly(
                smoothed_mask, _ratio.numerator, _ratio.denominator
            )
        )
        * _trim_frac
        / 2
    )
)

# Reference: downsampled + trimmed original mask
smoothed_mask_trimmed = _downsample_trim(smoothed_mask.astype(float), _ratio, _trim_n)

# ── Design the same Butterworth filter used in the pipeline ──────────────────
if _lowpass is not None:
    _sos = scipy_signal.butter(
        _order, [_highpass, _lowpass], btype="bandpass", fs=_fs, output="sos"
    )
else:
    _sos = scipy_signal.butter(
        _order, _highpass, btype="highpass", fs=_fs, output="sos"
    )

# ── Filter the gap indicator (1 - mask) ──────────────────────────────────────
_gap_indicator = 1.0 - smoothed_mask.astype(float)
_gap_filtered = scipy_signal.sosfiltfilt(_sos, _gap_indicator)

gap_contamination = _downsample_trim(_gap_filtered, _ratio, _trim_n)

# ── Build extended mask ───────────────────────────────────────────────────────
contamination_threshold = 0.01

_original_gap = smoothed_mask_trimmed < 0.5
_contaminated = np.abs(gap_contamination) > contamination_threshold
_excluded = _original_gap | _contaminated

# Merge excluded regions separated by a short clean gap
min_clean_hours = 12
min_clean_samples = int(min_clean_hours * 3600 * _target_fs)
_excluded = binary_closing(_excluded, structure=np.ones(min_clean_samples))

extended_mask_binary = (~_excluded).astype(float)


# ── Match length to sp_0.N ────────────────────────────────────────────────────
# resample_poly rounding can produce a 1-2 sample difference vs the pipeline.
# Clip or pad to sp_0.N so masks broadcast correctly against sp_0.data[ch].
def _match_length(arr, target_n):
    if len(arr) >= target_n:
        return arr[:target_n]
    return np.pad(arr, (0, target_n - len(arr)), constant_values=arr[-1])


smoothed_mask_trimmed = _match_length(smoothed_mask_trimmed, sp_0.N)
gap_contamination = _match_length(gap_contamination, sp_0.N)
extended_mask_binary = _match_length(extended_mask_binary, sp_0.N)

_orig_gap = 1 - smoothed_mask_trimmed.mean()
_ext_gap = 1 - extended_mask_binary.mean()
print(f"Original gap fraction : {_orig_gap:.4%}")
print(
    f"Extended gap fraction : {_ext_gap:.4%}  (+{_ext_gap - _orig_gap:.4%} from filter corruption)"
)
print(f"Mask length: {len(extended_mask_binary):,}  sp_0.N: {sp_0.N:,}  ✓")

In [ ]:
# ── Apply half-cosine taper at each gap edge ─────────────────────────────────
taper_seconds = 12 * 3600  # ramp length on each side of a gap edge (seconds)
taper_samples = int(taper_seconds * sp_0.fs)


def taper_mask_edges(binary_mask, taper_n):
    """Raise-cosine taper at every 0→1 (rising) and 1→0 (falling) edge."""
    mask = binary_mask.copy()
    N = len(mask)
    diff = np.diff(np.concatenate([[0], binary_mask.astype(int), [0]]))
    half_hann = 0.5 * (1 - np.cos(np.pi * np.arange(taper_n) / taper_n))

    for r in np.where(diff == 1)[0]:  # rising edges (gap → data)
        end = min(r + taper_n, N)
        n = end - r
        mask[r:end] *= half_hann[:n]

    for f in np.where(diff == -1)[0]:  # falling edges (data → gap)
        start = max(f - taper_n, 0)
        n = f - start
        mask[start:f] *= half_hann[:n][::-1]

    return mask


gap_mask_tapered = taper_mask_edges(extended_mask_binary, taper_samples)

# ── Taper the dataset left and right endpoints ────────────────────────────────
# Apply an independent cosine ramp to the very start and end of the full window
# so the dataset endpoints are smoothly brought to zero regardless of gaps.
edge_taper_seconds = 12 * 3600  # 4-hour ramp at each end of the dataset
edge_taper_n = int(edge_taper_seconds * sp_0.fs)
_edge_ramp = 0.5 * (1 - np.cos(np.pi * np.arange(edge_taper_n) / edge_taper_n))

gap_mask_tapered[:edge_taper_n] *= _edge_ramp
gap_mask_tapered[-edge_taper_n:] *= _edge_ramp[::-1]

print(f"Gap-edge taper : {taper_seconds / 3600:.1f} h per gap edge")
print(f"Dataset-end taper: {edge_taper_seconds / 3600:.1f} h each side")

# ── Visualise around the first gap ───────────────────────────────────────────
t_days = (sp_0.t - sp_0.t[0]) / 86400

_diff = np.diff(np.concatenate([[1], extended_mask_binary.astype(int), [1]]))
_gap_start = np.where(_diff == -1)[0]
_gap_end = np.where(_diff == 1)[0]

if len(_gap_start):
    _gc = (_gap_start[0] + _gap_end[0]) // 2
    _zoom = int(3 * 86400 * sp_0.fs)
else:
    _gc, _zoom = sp_0.N // 2, sp_0.N // 4

_lo, _hi = max(0, _gc - _zoom), min(sp_0.N, _gc + _zoom)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(
    t_days[_lo:_hi],
    smoothed_mask_trimmed[_lo:_hi],
    label="Original mask (downsampled + trimmed)",
    alpha=0.7,
)
axes[0].plot(
    t_days[_lo:_hi],
    gap_contamination[_lo:_hi],
    label="Filter leakage  [filtered(1 − mask)]",
    alpha=0.9,
)
axes[0].axhline(
    contamination_threshold,
    color="k",
    ls="--",
    label=f"±threshold = {contamination_threshold}",
)
axes[0].axhline(-contamination_threshold, color="k", ls="--")
axes[0].set_ylabel("Amplitude")
axes[0].legend(fontsize=11)
axes[0].set_title("Butterworth filter leakage outside gap edges")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    t_days[_lo:_hi],
    extended_mask_binary[_lo:_hi],
    label="Extended binary mask",
    alpha=0.8,
)
axes[1].plot(
    t_days[_lo:_hi],
    gap_mask_tapered[_lo:_hi],
    label=f"Final window  (gap taper={taper_seconds/3600:.0f} h, "
    f"edge taper={edge_taper_seconds/3600:.0f} h)",
    alpha=0.9,
)
axes[1].set_ylabel("Mask value")
axes[1].set_xlabel("Time [days]")
axes[1].legend(fontsize=11)
axes[1].set_title("Final window function")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
t_days = (sp_0.data["t"] - sp_0.data["t"][0]) / 86400  # relative time in days

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

for i, ch in enumerate(["X", "Y", "Z"]):
    axes[i].plot(t_days, sp_0.data[ch], linewidth=0.4, color=f"C{i}", label=f"TDI-{ch}")
    axes[i].set_ylabel("Frac. freq.", fontsize=12)
    axes[i].legend(loc="upper right", fontsize=11)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_title(f"TDI-{ch}", fontsize=12)
    axes[i].set_xlim([6, 14])

axes[2].set_xlabel("Time [days]", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
t_days = (sp_0.data["t"] - sp_0.data["t"][0]) / 86400  # relative time in days

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

for i, ch in enumerate(["X", "Y", "Z"]):
    axes[i].plot(
        t_days,
        sp_0.data[ch] * gap_mask_tapered,
        linewidth=0.4,
        color=f"C{i}",
        label=f"TDI-{ch}",
    )
    axes[i].set_ylabel("Frac. freq.", fontsize=12)
    axes[i].legend(loc="upper right", fontsize=11)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_title(f"TDI-{ch}", fontsize=12)
    axes[i].set_xlim([0, 1])

axes[2].set_xlabel("Time [days]", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply gap mask to XYZ channels ───────────────────────────────────────────
for ch in sp_0.channels:
    sp_0._data[ch] = sp_0._data[ch] * gap_mask_tapered

# ── Derive AET from the masked XYZ ───────────────────────────────────────────
sp_0_aet = sp_0.to_aet()

print(f"Gap mask applied to: {sp_0.channels}")
print(f"AET derived from masked XYZ: {sp_0_aet.channels}")

## 3. Compute FFT and Periodogram

The one-sided periodogram estimate of the noise Power Spectral Density $S$ is

$$\hat{S}(f_k) = \frac{2\,\Delta t}{N} \left|\tilde{n}(f_k)\right|^2$$

where $\tilde{n}$ is the FFT of the processed time series, $\Delta t$ the sampling interval and $N$ the length of the truncated data set.

In [ ]:
# Data is already normalised by laser_frequency inside process_pipeline.
# Use the new periodogram() and to_aet() methods directly.
CENTRAL_FREQ = data["metadata"]["laser_frequency"]

freq, fft_xyz = sp_0.fft()

sp_0_aet = sp_0.to_aet()
_, fft_aet = sp_0_aet.fft()

# psd_norm for reference (this is the factor baked into periodogram())
psd_norm = 2 * sp_0.dt / sp_0.N

psd_xyz = {ch: (np.abs(fft_xyz[ch]) ** 2) * psd_norm for ch in ["X", "Y", "Z"]}
psd_aet = {ch: (np.abs(fft_aet[ch]) ** 2) * psd_norm for ch in ["A", "E", "T"]}

# Mojito L1 noise estimates, normalised to fractional frequency units
noise_freqs = data["noise_estimates"]["freqs"]
noise_cov_xyz = data["noise_estimates"]["xyz"]
noise_cov_aet = data["noise_estimates"]["aet"]

l1_xyz = {
    ch: noise_cov_xyz[0][:, i, i] / CENTRAL_FREQ**2
    for i, ch in enumerate(["X", "Y", "Z"])
}
l1_aet = {
    ch: noise_cov_aet[0][:, i, i] / CENTRAL_FREQ**2
    for i, ch in enumerate(["A", "E", "T"])
}

## 4. Periodogram vs Mojito L1 Estimate

A good processing pipeline produces a periodogram (coloured lines) that traces the Mojito L1 noise model (red dashed) throughout the science band (1×10⁻⁴ to 1×10⁻¹ Hz). Potential deviations at the lowest frequencies indicate residual artefacts from filtering or insufficient trimming.

In [ ]:
nyquist = sp_0.fs / 2
xyz_colors = ["C0", "C1", "C2"]
aet_colors = ["#e377c2", "#9467bd", "#8c564b"]

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=False, sharey=True)

for i, ch in enumerate(["X", "Y", "Z"]):
    ax = axes[i, 0]
    ax.loglog(
        freq[1:],
        psd_xyz[ch][1:],
        linewidth=1.0,
        color=xyz_colors[i],
        label=f"TDI-{ch}",
    )
    ax.loglog(
        noise_freqs,
        l1_xyz[ch],
        linestyle="dashed",
        color="red",
        linewidth=2.0,
        label="Mojito L1 estimate",
    )
    ax.axvspan(1e-4, 1e-1, alpha=0.15, color="green", label="Science band")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.set_ylabel("PSD [1/Hz]", fontsize=20)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left", fontsize=14, framealpha=0.95)
    ax.tick_params(axis="both", which="major", labelsize=16)
for i, ch in enumerate(["A", "E", "T"]):
    ax = axes[i, 1]
    ax.loglog(
        freq[1:],
        psd_aet[ch][1:],
        linewidth=1.0,
        color=aet_colors[i],
        label=f"TDI-{ch}",
    )
    ax.loglog(noise_freqs, l1_aet[ch], linestyle="dashed", color="red", linewidth=2.0)
    ax.axvspan(1e-4, 1e-1, alpha=0.15, color="green")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left", fontsize=14, framealpha=0.95)
    ax.tick_params(axis="both", which="major", labelsize=16)

axes[2, 0].set_xlabel("Frequency [Hz]", fontsize=20)
axes[2, 1].set_xlabel("Frequency [Hz]", fontsize=20)

fig.suptitle(
    f"Processed Mojito Periodogram vs L1 Estimate (segment0)\n"
    f"HP={filter_kwargs['highpass_cutoff']:.0e} Hz, "
    f"LP={filter_kwargs['lowpass_cutoff']} Hz, "
    f"trim={trim_kwargs['fraction']:.1%}, "
    f"{truncate_kwargs['days']} days/segment, "
    f"fs={sp_0.fs} Hz, "
    f"window={window_kwargs['window']} α={window_kwargs.get('alpha', 'N/A')}",
    fontsize=20,
    fontweight="bold",
)
plt.tight_layout()
plt.show()